# EP11. 종합 프로젝트: 사규 마스터 에이전트

## 학습 목표

1. **Hybrid Search** (BM25 + ChromaDB)로 사규 문서를 정확하게 검색한다
2. **Reranking** (CohereRerank / BGE)으로 검색 정확도를 추가로 향상시킨다
3. **deepagents** (`create_deep_agent`)로 멀티스텝 질의응답 에이전트를 구축한다
4. **Langfuse**로 에이전트 품질을 모니터링하고 A/B 테스트를 수행한다

## AI 가이드

> 막히는 부분은 Claude에게 물어보세요.
> 예시 프롬프트:
> - "EnsembleRetriever에서 BM25와 벡터 검색 가중치를 어떻게 조정해야 해?"
> - "CohereRerank 대신 BGE cross-encoder를 사용하는 코드를 작성해줘"
> - "create_deep_agent에서 subagents 파라미터는 어떻게 사용해?"
> - "Langfuse score API로 여러 지표를 동시에 기록하는 방법을 알려줘"

**사전 요구사항**: EP06 (Langfuse 기초), EP10 (멀티에이전트 기초)

**예상 소요 시간**: 120~150분

**필요한 API 키**:
- `ANTHROPIC_API_KEY`
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`
- `COHERE_API_KEY` (선택: Reranking, 없으면 BGE 사용)

**전체 아키텍처**:

```mermaid
flowchart TD
    A[사규 문서] --> B(PyMuPDF / TextSplitter)
    B --> C[(BM25 키워드 🚀)]
    B --> D[(ChromaDB 벡터 🧠)]
    C --> E{EnsembleRetriever}
    D --> E
    E --> F(Reranking ✨)
    F --> G(deepagents 에이전트 🤖)
    G --> H([최종 답변 🎯])
    G -.-> I(Langfuse 모니터링 📊)
```

사규 텍스트 → 청크 분할 → BM25 인덱스
                       → ChromaDB 벡터 인덱스
                              ↓
              EnsembleRetriever (Hybrid Search)
                              ↓
                     Reranking (상위 3개)
                              ↓
              deepagents 에이전트 (Claude)
                              ↓
              Langfuse 모니터링 & A/B 테스트
```


## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langchain langchain-anthropic langchain-community langchain-cohere \
    langgraph deepagents langfuse chromadb rank-bm25 pymupdf python-dotenv \
    sentence-transformers

## 섹션 2. 라이브러리 로드 + Langfuse 초기화

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 설정 (또는 .env 파일 사용)
# os.environ["ANTHROPIC_API_KEY"] = "your-key"
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"
# os.environ["COHERE_API_KEY"] = "your-cohere-key"  # 선택

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

# Langfuse 초기화
langfuse = Langfuse(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
)

try:
    langfuse.auth_check()
    print("Langfuse 연결 성공!")
except Exception as e:
    print(f"Langfuse 연결 실패 (학습은 계속 가능): {e}")

# LLM 초기화
llm = ChatAnthropic(model="claude-opus-4-5", temperature=0)
print("라이브러리 로드 완료!")

## 섹션 3. 샘플 사규 문서 준비

실제 PDF 대신, 학습용으로 연차·복무·보안 규정 등 인라인 샘플 사규를 사용합니다.

In [ ]:
# 샘플 사규 문서 정의 (인라인 텍스트)
SAMPLE_COMPANY_POLICIES = [
    {
        "title": "제1조 연차 휴가 규정",
        "content": """
        제1조 (연차 유급휴가)
        1항: 1년 이상 근속한 직원은 연간 15일의 연차 유급휴가를 받는다.
        2항: 3년 이상 근속 시 매 2년마다 1일씩 추가되며, 최대 25일을 한도로 한다.
        3항: 입사 1년 미만 직원은 매월 개근 시 1일의 유급휴가를 받는다.
        4항: 연차휴가는 반드시 당해 연도에 사용하여야 하며, 미사용 시 소멸된다.
        5항: 불가피한 사유로 미사용한 연차는 연 최대 5일까지 다음 연도로 이월할 수 있다.
        6항: 연차 사용은 최소 3일 전에 결재 라인을 통해 신청하여야 한다.
        """,
        "source": "인사규정",
        "page": 5,
    },
    {
        "title": "제2조 출산 및 육아 휴직 규정",
        "content": """
        제2조 (출산전후휴가 및 육아휴직)
        1항: 출산전후휴가는 출산일 전후로 90일을 부여하며, 출산 후 45일 이상이어야 한다.
        2항: 다태아(쌍둥이 이상) 출산 시 120일의 휴가를 부여한다.
        3항: 육아휴직은 만 8세 이하 또는 초등학교 2학년 이하 자녀를 둔 직원이 신청할 수 있다.
        4항: 육아휴직 기간은 최대 1년이며, 부모가 각각 사용할 수 있다.
        5항: 육아휴직 기간 중 통상임금의 80%(상한 150만원)를 지급한다.
        6항: 배우자 출산휴가는 10일을 부여하며, 출산일로부터 90일 이내에 사용하여야 한다.
        """,
        "source": "인사규정",
        "page": 8,
    },
    {
        "title": "제3조 복무 규정 (근무 시간)",
        "content": """
        제3조 (근무시간 및 복무)
        1항: 표준 근무시간은 주 40시간으로, 월요일~금요일 09:00~18:00이다.
        2항: 점심시간은 12:00~13:00 (1시간)이다.
        3항: 유연근무제를 신청한 직원은 07:00~10:00 사이에 출근할 수 있다.
        4항: 재택근무는 팀장 승인 후 주 최대 2일까지 가능하다.
        5항: 연장근무는 사전에 팀장 승인을 받아야 하며, 월 최대 52시간을 초과할 수 없다.
        6항: 지각 3회는 경고 1회로 처리하며, 경고 3회 누적 시 인사위원회에 회부된다.
        """,
        "source": "복무규정",
        "page": 12,
    },
    {
        "title": "제4조 보안 정책 (정보보안)",
        "content": """
        제4조 (정보보안 정책)
        1항: 회사의 모든 기밀 정보는 대외비로 취급하며, 무단 유출 시 징계 처분한다.
        2항: 개인 USB 저장장치는 IT 보안팀의 사전 승인 없이 업무용 PC에 연결할 수 없다.
        3항: 업무용 PC 비밀번호는 90일마다 변경하여야 하며, 이전 5개 비밀번호 재사용 불가.
        4항: 퇴근 시 PC 전원을 완전히 종료하고 화면 잠금을 설정하여야 한다.
        5항: 사내 네트워크에서 SNS, 스트리밍 서비스 등 업무 무관 사이트 접속은 금지한다.
        6항: 업무용 이메일로 개인 파일을 외부로 전송하는 행위는 보안감사 대상이다.
        7항: 보안 사고 발생 즉시 IT 보안팀(내선 1234)에 신고하여야 한다.
        """,
        "source": "보안정책",
        "page": 3,
    },
    {
        "title": "제5조 경비 및 비용 처리 규정",
        "content": """
        제5조 (업무 경비 처리)
        1항: 업무상 발생한 경비는 지출일로부터 30일 이내에 증빙을 첨부하여 신청하여야 한다.
        2항: 식대는 1인당 1식 최대 15,000원, 접대비는 1인당 50,000원 한도로 처리한다.
        3항: 100만원 이상 경비는 팀장 및 부서장 결재가 모두 필요하다.
        4항: 해외 출장 경비는 출장 전 예산 신청 후 사후 정산하는 방식으로 처리한다.
        5항: 개인카드 사용 후 정산 시에는 법인카드 영수증에 준하는 증빙을 제출하여야 한다.
        6항: 명백한 사적 목적의 지출은 경비로 처리할 수 없으며, 적발 시 전액 반납한다.
        """,
        "source": "경비규정",
        "page": 22,
    },
    {
        "title": "제6조 성과 평가 및 승진 기준",
        "content": """
        제6조 (성과평가 및 승진)
        1항: 성과평가는 연 2회(6월, 12월) 실시하며, S·A·B·C·D 5등급으로 평가한다.
        2항: 승진은 현 직급 최소 근속 기간(주임 2년, 대리 3년, 과장 4년)을 충족하여야 한다.
        3항: 직전 2회 연속 B등급 이상을 받아야 승진 심사 대상이 된다.
        4항: D등급을 2회 연속 받은 직원은 성과개선계획(PIP)에 따라 관리된다.
        5항: 성과급은 연봉의 0~30% 범위에서 등급에 따라 차등 지급한다.
        """,
        "source": "인사규정",
        "page": 18,
    },
    {
        "title": "제7조 교육 훈련 및 자기개발 지원",
        "content": """
        제7조 (교육훈련 및 자기개발 지원)
        1항: 회사는 직원의 역량 개발을 위해 연간 1인당 최대 200만원의 교육비를 지원한다.
        2항: 자격증 취득 시 업무 관련성에 따라 취득 비용의 50~100%를 지원한다.
        3항: 직무 관련 온라인 강의 수강료는 영수증 제출 후 100% 환급한다.
        4항: 대학원 진학 시 사전 승인 후 등록금의 50%를 지원하며, 수료 후 2년 재직 의무가 있다.
        5항: 사내 도서 구매 신청은 월 3만원 한도로 팀장 승인 후 가능하다.
        """,
        "source": "인사규정",
        "page": 25,
    },
]

print(f"사규 문서 {len(SAMPLE_COMPANY_POLICIES)}개 준비 완료")
for p in SAMPLE_COMPANY_POLICIES:
    print(f"  - [{p['source']} p.{p['page']}] {p['title']}")

## 섹션 4. 문서 분할 (RecursiveCharacterTextSplitter)

각 조항을 청크로 분할하고 LangChain `Document` 형식으로 변환합니다.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# 한국어에 최적화된 텍스트 분할기
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len,
)

# 사규 텍스트를 LangChain Document로 변환
raw_documents = []
for policy in SAMPLE_COMPANY_POLICIES:
    doc = Document(
        page_content=f"{policy['title']}\n{policy['content'].strip()}",
        metadata={
            "source": policy["source"],
            "page": policy["page"],
            "title": policy["title"],
        },
    )
    raw_documents.append(doc)

# 청크 분할
split_documents = splitter.split_documents(raw_documents)

print(f"원본 문서: {len(raw_documents)}개")
print(f"분할 후 청크: {len(split_documents)}개")
print(f"\n예시 청크 (첫 번째):")
print(f"  내용: {split_documents[0].page_content[:150]}...")
print(f"  메타데이터: {split_documents[0].metadata}")

## 섹션 5. BM25 인덱스 구축

BM25는 TF-IDF 기반의 키워드 검색 알고리즘으로, 정확한 조항 번호나 법적 용어 검색에 강합니다.

In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25 인덱스 구축
bm25_retriever = BM25Retriever.from_documents(
    split_documents,
    k=5,  # 상위 5개 반환
)
bm25_retriever.k = 5

# BM25 검색 테스트
bm25_results = bm25_retriever.invoke("연차 휴가 일수")
print("=== BM25 검색 결과: '연차 휴가 일수' ===")
for i, doc in enumerate(bm25_results[:3], 1):
    print(f"\n[{i}] [{doc.metadata['source']} p.{doc.metadata['page']}]")
    print(f"     {doc.page_content[:120]}...")

print(f"\nBM25 인덱스 구축 완료: {len(split_documents)}개 청크 인덱싱")

## 섹션 6. 벡터 인덱스 구축 (ChromaDB + HuggingFaceEmbeddings)

벡터 검색은 의미론적 유사성 기반으로, 동의어나 의미가 같은 다른 표현을 검색할 때 강합니다.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("임베딩 모델 로드 중... (첫 실행 시 다운로드로 수 분 소요)")

# 한국어 지원 다국어 임베딩 모델
embeddings = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# ChromaDB 벡터 인덱스 구축
vectorstore = Chroma.from_documents(
    documents=split_documents,
    embedding=embeddings,
    collection_name="company_policy",
    persist_directory="./chroma_db",  # 영속성 저장
)

vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# 벡터 검색 테스트
vector_results = vector_retriever.invoke("육아 관련 쉬는 날")
print("\n=== 벡터 검색 결과: '육아 관련 쉬는 날' ===")
for i, doc in enumerate(vector_results[:3], 1):
    print(f"\n[{i}] [{doc.metadata['source']} p.{doc.metadata['page']}]")
    print(f"     {doc.page_content[:120]}...")

print(f"\nChromaDB 벡터 인덱스 구축 완료")

## 섹션 7. EnsembleRetriever로 Hybrid Search 구성

BM25(키워드)와 벡터(의미)를 결합하여 두 방식의 장점을 모두 활용합니다.

In [ ]:
from langchain.retrievers import EnsembleRetriever

# EnsembleRetriever: BM25 40% + 벡터 60%
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

# Hybrid Search 테스트
print("=== Hybrid Search 비교 ===")

test_queries = [
    "연차를 몇 일 쓸 수 있나요?",
    "보안 규정 위반 시 어떻게 되나요?",
    "재택근무 신청 방법",
]

for query in test_queries:
    results = ensemble_retriever.invoke(query)
    print(f"\n쿼리: '{query}'")
    print(f"검색 결과 {len(results)}개:")
    for i, doc in enumerate(results[:2], 1):
        print(f"  [{i}] [{doc.metadata['source']} p.{doc.metadata['page']}] "
              f"{doc.page_content[:80]}...")

print("\nEnsembleRetriever 구성 완료 (BM25 40% + 벡터 60%)")

## 섹션 8. Reranking 통합

검색된 후보 문서를 쿼리와의 관련성 기준으로 재순위화합니다.
Cohere API 키가 있으면 CohereRerank, 없으면 BGE cross-encoder를 사용합니다.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever

USE_COHERE = bool(os.environ.get("COHERE_API_KEY"))

if USE_COHERE:
    print("Cohere Rerank 사용")
    from langchain_cohere import CohereRerank
    compressor = CohereRerank(
        model="rerank-multilingual-v3.0",
        top_n=3,
        cohere_api_key=os.environ["COHERE_API_KEY"],
    )
else:
    print("BGE Cross-Encoder Rerank 사용 (COHERE_API_KEY 없음)")
    from langchain.retrievers.document_compressors import CrossEncoderReranker
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder

    cross_encoder = HuggingFaceCrossEncoder(
        model_name="BAAI/bge-reranker-v2-m3"
    )
    compressor = CrossEncoderReranker(
        model=cross_encoder,
        top_n=3,
    )

# Reranking Retriever 구성
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)

# Reranking 효과 비교
test_query = "출산 후 사용할 수 있는 휴가가 얼마나 되나요?"

print(f"\n쿼리: '{test_query}'")

print("\n[Hybrid Search만 (Reranking 전)]")
hybrid_results = ensemble_retriever.invoke(test_query)
for i, doc in enumerate(hybrid_results[:3], 1):
    print(f"  [{i}] [{doc.metadata['source']} p.{doc.metadata['page']}] "
          f"{doc.page_content[:100]}...")

print("\n[Reranking 후]")
reranked_results = reranking_retriever.invoke(test_query)
for i, doc in enumerate(reranked_results, 1):
    print(f"  [{i}] [{doc.metadata['source']} p.{doc.metadata['page']}] "
          f"{doc.page_content[:100]}...")

print("\nReranking 통합 완료!")

## 섹션 9. deepagents로 질의응답 에이전트 구축

`create_deep_agent`를 사용하여 멀티스텝 추론이 가능한 사규 에이전트를 구축합니다.

In [ ]:
from deepagents import create_deep_agent

# 사규 검색 도구 정의
@tool
def search_company_policy(query: str) -> str:
    """
    회사 사규(취업규칙, 인사규정, 복무규정, 보안정책, 경비규정)에서
    관련 조항을 검색합니다.
    반드시 구체적인 질문이나 키워드를 입력하세요.
    """
    docs = reranking_retriever.invoke(query)
    if not docs:
        return "관련 조항을 찾지 못했습니다. 다른 키워드로 검색해보세요."

    results = []
    for doc in docs:
        source = doc.metadata.get("source", "미상")
        page = doc.metadata.get("page", "?")
        results.append(f"[출처: {source} p.{page}]\n{doc.page_content}")

    return "\n\n".join(results)

@tool
def compare_policies(policy_a: str, policy_b: str) -> str:
    """
    두 개의 사규 항목을 비교합니다.
    예: '연차 휴가'와 '육아휴직'을 비교하려면
    policy_a='연차 휴가', policy_b='육아휴직'으로 입력하세요.
    """
    results_a = reranking_retriever.invoke(policy_a)
    results_b = reranking_retriever.invoke(policy_b)

    output = []
    output.append(f"=== {policy_a} ===")
    for doc in results_a[:2]:
        output.append(doc.page_content[:200])

    output.append(f"\n=== {policy_b} ===")
    for doc in results_b[:2]:
        output.append(doc.page_content[:200])

    return "\n".join(output)

# deepagents 에이전트 생성
policy_agent = create_deep_agent(
    model=llm,
    tools=[search_company_policy, compare_policies],
    system_prompt="""당신은 회사 사규 전문 AI 어시스턴트입니다.

역할:
- 직원들의 사규 관련 질문에 정확하게 답변합니다.
- 반드시 search_company_policy 도구로 조항을 검색한 후 답변합니다.
- 출처(문서명, 페이지)를 항상 명시합니다.
- 복잡한 질문은 여러 번 검색하여 완전한 답변을 제공합니다.

답변 형식:
1. 핵심 답변 (1~2문장)
2. 상세 설명 (관련 조항 인용)
3. 출처: [문서명 페이지]

주의: 검색 결과에 없는 내용은 절대 추측하지 마세요.""",
)

print("deepagents 에이전트 구축 완료!")

# 기본 테스트
test_result = policy_agent.invoke(
    {"messages": [HumanMessage(content="연차는 최대 몇 일인가요?")]}
)
print("\n=== 에이전트 응답 ===")
print(test_result["messages"][-1].content)

## 섹션 10. Langfuse 통합

`CallbackHandler`를 주입하여 모든 에이전트 실행을 자동으로 트레이싱합니다.

In [ ]:
def create_handler(session_id: str = "ep11", tags: list = None) -> CallbackHandler:
    """Langfuse CallbackHandler 생성 헬퍼"""
    return CallbackHandler(
        public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
        host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
        session_id=session_id,
        user_id="ep11-student",
        tags=tags or ["ep11", "company-policy"],
    )

def ask_policy_agent(
    query: str,
    session_id: str = "ep11-main",
    tags: list = None,
) -> dict:
    """Langfuse 트레이싱이 포함된 에이전트 실행"""
    handler = create_handler(session_id=session_id, tags=tags)

    result = policy_agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config={
            "callbacks": [handler],
            "recursion_limit": 20,
        },
    )
    answer = result["messages"][-1].content
    trace_id = handler.get_trace_id()

    return {"answer": answer, "trace_id": trace_id, "handler": handler}

# Langfuse 통합 테스트
print("Langfuse 통합 테스트 실행 중...")
result = ask_policy_agent(
    "출산 후 사용할 수 있는 휴가 종류와 기간을 알려주세요.",
    session_id="ep11-langfuse-test",
    tags=["ep11", "integration-test"],
)

print(f"Trace ID: {result['trace_id']}")
print(f"\n=== 응답 ===")
print(result["answer"])
print(f"\nLangfuse 대시보드에서 트레이스를 확인하세요!")

## 섹션 11. 엔드투엔드 파이프라인 테스트

다양한 유형의 질문으로 전체 파이프라인을 테스트합니다.

In [ ]:
# 다양한 질문 유형 테스트
E2E_TEST_CASES = [
    {
        "query": "입사 6개월 됐는데 연차를 쓸 수 있나요?",
        "expected_keywords": ["월", "개근", "1일"],
        "category": "연차",
    },
    {
        "query": "재택근무를 주 3일 하고 싶은데 가능한가요?",
        "expected_keywords": ["2일", "팀장"],
        "category": "복무",
    },
    {
        "query": "비밀번호를 얼마나 자주 바꿔야 하나요?",
        "expected_keywords": ["90일"],
        "category": "보안",
    },
    {
        "query": "대리에서 과장으로 승진하려면 무엇이 필요한가요?",
        "expected_keywords": ["3년", "B등급"],
        "category": "승진",
    },
    {
        "query": "외부 교육 비용을 회사에서 지원받을 수 있나요?",
        "expected_keywords": ["200만원", "자격증"],
        "category": "교육",
    },
]

print("=" * 60)
print("엔드투엔드 파이프라인 테스트")
print("=" * 60)

e2e_results = []
for i, tc in enumerate(E2E_TEST_CASES, 1):
    print(f"\n[{i}/{len(E2E_TEST_CASES)}] 카테고리: {tc['category']}")
    print(f"질문: {tc['query']}")

    result = ask_policy_agent(
        tc["query"],
        session_id="ep11-e2e-test",
        tags=["ep11", "e2e-test", tc["category"]],
    )

    answer = result["answer"]
    keywords_found = [
        kw for kw in tc["expected_keywords"]
        if kw in answer
    ]
    accuracy = len(keywords_found) / len(tc["expected_keywords"])

    print(f"응답 (앞 150자): {answer[:150]}...")
    print(f"핵심 키워드 포함: {keywords_found} / {tc['expected_keywords']}")
    print(f"정확도: {accuracy:.0%}")

    e2e_results.append({
        "category": tc["category"],
        "trace_id": result["trace_id"],
        "accuracy": accuracy,
    })

avg_accuracy = sum(r["accuracy"] for r in e2e_results) / len(e2e_results)
print(f"\n전체 평균 정확도: {avg_accuracy:.1%}")

## 섹션 12. 평가: Langfuse score API로 정확도 기록

각 응답의 품질 점수를 Langfuse에 기록하여 시간에 따른 품질 추이를 관찰합니다.

In [ ]:
def record_quality_scores(result_list: list):
    """E2E 테스트 결과를 Langfuse에 score로 기록"""
    recorded = 0
    for result in result_list:
        try:
            langfuse.score(
                trace_id=result["trace_id"],
                name="answer_accuracy",
                value=result["accuracy"],
                data_type="NUMERIC",
                comment=f"카테고리: {result['category']}, 키워드 기반 정확도",
            )
            recorded += 1
        except Exception as e:
            print(f"score 기록 실패 ({result['category']}): {e}")

    return recorded

# Langfuse에 품질 점수 기록
print("Langfuse에 품질 점수 기록 중...")
recorded_count = record_quality_scores(e2e_results)
print(f"{recorded_count}/{len(e2e_results)}개 score 기록 완료")

# 결과 요약 테이블 출력
print("\n=== 품질 평가 요약 ===")
print(f"{'카테고리':<12} {'정확도':<10} {'Trace ID (앞 8자)'}")
print("-" * 40)
for r in e2e_results:
    tid = r['trace_id'][:8] if r['trace_id'] else 'N/A'
    print(f"{r['category']:<12} {r['accuracy']:<10.0%} {tid}...")

avg = sum(r['accuracy'] for r in e2e_results) / len(e2e_results)
print("-" * 40)
print(f"{'평균':<12} {avg:<10.1%}")

## 섹션 13. A/B 테스트: Reranking 유무 비교

`tags`를 활용하여 Reranking이 없는 버전(Variant A)과 있는 버전(Variant B)을 비교합니다.

In [ ]:
# A/B 테스트용 에이전트 변형 구성
# Variant A: Hybrid Search만 (Reranking 없음)
@tool
def search_policy_no_rerank(query: str) -> str:
    """사규에서 관련 조항을 검색합니다 (Reranking 미적용)."""
    docs = ensemble_retriever.invoke(query)
    if not docs:
        return "관련 조항을 찾지 못했습니다."
    results = []
    for doc in docs[:3]:
        source = doc.metadata.get("source", "미상")
        page = doc.metadata.get("page", "?")
        results.append(f"[출처: {source} p.{page}]\n{doc.page_content}")
    return "\n\n".join(results)

agent_variant_a = create_deep_agent(
    model=llm,
    tools=[search_policy_no_rerank],
    system_prompt="""당신은 회사 사규 전문 AI입니다.
search_policy_no_rerank 도구로 검색 후 출처를 명시하여 답변하세요.""",
)

# Variant B: Hybrid Search + Reranking (이미 구축된 policy_agent 사용)
# policy_agent = Variant B

print("A/B 테스트 설정 완료")
print("  Variant A: EnsembleRetriever만 (Reranking 없음)")
print("  Variant B: EnsembleRetriever + Reranking")

In [ ]:
# A/B 테스트 실행
AB_TEST_QUERIES = [
    ("쌍둥이 출산 시 휴가는 며칠인가요?", ["120일"]),
    ("USB를 PC에 꽂아도 되나요?", ["승인", "금지"]),
    ("교육비 지원 한도가 얼마인가요?", ["200만원"]),
    ("연장근무는 한 달에 최대 얼마나 가능한가요?", ["52시간"]),
    ("영수증 없이 경비 처리할 수 있나요?", ["증빙", "반납"]),
]

ab_results = {"variant_a": [], "variant_b": []}

for i, (query, expected_kws) in enumerate(AB_TEST_QUERIES, 1):
    print(f"\n[{i}/{len(AB_TEST_QUERIES)}] 쿼리: {query}")

    for variant_name, agent_variant, variant_tag in [
        ("variant_a", agent_variant_a, "ab-variant-a-no-rerank"),
        ("variant_b", policy_agent, "ab-variant-b-reranking"),
    ]:
        handler = create_handler(
            session_id=f"ep11-ab-{variant_name}",
            tags=["ep11", "ab-test", variant_tag],
        )
        try:
            result = agent_variant.invoke(
                {"messages": [HumanMessage(content=query)]},
                config={"callbacks": [handler], "recursion_limit": 20},
            )
            answer = result["messages"][-1].content
        except Exception as e:
            answer = f"ERROR: {e}"

        kws_found = [kw for kw in expected_kws if kw in answer]
        accuracy = len(kws_found) / len(expected_kws)

        trace_id = handler.get_trace_id()

        # Langfuse에 A/B 점수 기록
        try:
            langfuse.score(
                trace_id=trace_id,
                name="ab_answer_accuracy",
                value=accuracy,
                data_type="NUMERIC",
                comment=f"variant={'A (no-rerank)' if 'a' in variant_name else 'B (reranking)'}; "
                        f"keywords={kws_found}",
            )
        except Exception:
            pass

        ab_results[variant_name].append(accuracy)
        print(f"  {variant_name.upper()}: 정확도={accuracy:.0%} | 키워드={kws_found}")

# 최종 비교 결과
avg_a = sum(ab_results["variant_a"]) / len(ab_results["variant_a"])
avg_b = sum(ab_results["variant_b"]) / len(ab_results["variant_b"])
improvement = ((avg_b - avg_a) / (avg_a + 1e-9)) * 100

print("\n" + "=" * 50)
print("A/B 테스트 최종 결과")
print("=" * 50)
print(f"Variant A (Reranking 없음): 평균 정확도 {avg_a:.1%}")
print(f"Variant B (Reranking 적용): 평균 정확도 {avg_b:.1%}")
print(f"개선율: {improvement:+.1f}%")
print("\nLangfuse 대시보드에서 'ab-variant-a-no-rerank' vs 'ab-variant-b-reranking' 태그로 비교하세요!")

## 정리

이 노트북에서 구축한 것:

| 컴포넌트 | 구현 내용 |
|---------|----------|
| 문서 준비 | 인라인 사규 텍스트 + `RecursiveCharacterTextSplitter` |
| BM25 | `BM25Retriever` - 정확한 키워드/조항 번호 검색 |
| 벡터 검색 | `ChromaDB` + `HuggingFaceEmbeddings` - 의미 기반 검색 |
| Hybrid Search | `EnsembleRetriever` (BM25 40% + 벡터 60%) |
| Reranking | `CohereRerank` 또는 `BGE CrossEncoder` |
| 에이전트 | `create_deep_agent` - 멀티스텝 질의응답 |
| 모니터링 | `Langfuse CallbackHandler` + `score API` |
| A/B 테스트 | `tags`로 Reranking 유무 비교 |

## Exercise

### Exercise 1: 자신의 문서로 파이프라인 구축

**목표**: 직접 보유한 문서(PDF, 텍스트 등)로 전체 파이프라인을 처음부터 구축한다.

**단계**:
1. 본인의 텍스트 문서 5개 이상을 `SAMPLE_COMPANY_POLICIES` 형식으로 정의 (또는 PDF 파일 사용 시 PyMuPDF로 파싱)
2. `RecursiveCharacterTextSplitter`로 청크 분할 (chunk_size 조정 실험)
3. BM25 + ChromaDB 인덱스 구축
4. `EnsembleRetriever`로 Hybrid Search 구성
5. Reranking 통합 (Cohere 또는 BGE 선택)
6. `create_deep_agent`로 에이전트 구성 (system_prompt 맞춤 작성)
7. 5개 이상의 실제 질의로 테스트
8. Langfuse에서 트레이스 확인 및 `answer_accuracy` score 기록

**제출**: Langfuse 대시보드 스크린샷 + 테스트 질의 5개와 답변

In [ ]:
# Exercise 1: 여기에 코드를 작성하세요

# Step 1: 자신의 문서 정의
MY_DOCUMENTS = [
    # {"title": "...", "content": "...", "source": "...", "page": 1},
    # 최소 5개 이상 추가하세요
]

# Step 2~3: 분할 및 인덱싱
# TODO

# Step 4: EnsembleRetriever
# TODO

# Step 5: Reranking
# TODO

# Step 6: 에이전트
# TODO

# Step 7~8: 테스트 및 Langfuse 기록
# TODO

### Exercise 2: Langfuse로 A/B 테스트 (EnsembleRetriever 가중치 비교)

**목표**: BM25와 벡터 검색의 가중치를 달리한 두 Variant를 Langfuse로 비교한다.

**단계**:
1. Variant A: `EnsembleRetriever(weights=[0.7, 0.3])` — BM25 우세
2. Variant B: `EnsembleRetriever(weights=[0.2, 0.8])` — 벡터 우세
3. 10개 테스트 쿼리 정의 (키워드 검색 유형 5개 + 의미 검색 유형 5개)
4. 두 Variant로 각각 실행 → `tags=["weight-a"]` / `tags=["weight-b"]`
5. 각 응답에 `retrieval_quality` score 기록 (0~1)
6. 결과 비교: 어떤 쿼리 유형에서 어느 가중치가 더 좋은가?

**힌트**: 조항 번호("제3조", "1항")가 포함된 질문은 BM25가 유리, 자연어 질문은 벡터가 유리

In [ ]:
# Exercise 2: 여기에 코드를 작성하세요

# Step 1~2: 두 가중치 Variant 구성
# variant_a_retriever = EnsembleRetriever(..., weights=[0.7, 0.3])
# variant_b_retriever = EnsembleRetriever(..., weights=[0.2, 0.8])
# TODO

# Step 3: 테스트 쿼리 정의 (키워드형 5개 + 자연어형 5개)
weight_test_queries = [
    # 키워드형 (BM25 유리)
    # ("제3조 복무 규정의 연장근무 한도", ["52시간"]),
    # ...
    # 자연어형 (벡터 유리)
    # ("회사를 오래 다니면 휴가가 늘어나나요?", ["25일"]),
    # ...
]

# Step 4~6: 실행, Langfuse 기록, 비교 분석
# TODO